In [ ]:
import os
import cv2
import math
import random
import PIL as pil
import numpy as np
from tqdm import tqdm
from itertools import product

import torch
import torchvision
from torchvision.utils import save_image
import torchvision.transforms as transforms
import torchvision.transforms.functional as F

from concurrent.futures import ThreadPoolExecutor

from albumentations import (
    Compose,
    HorizontalFlip,
    VerticalFlip,
    RandomBrightnessContrast,
    HueSaturationValue,
    GaussianBlur,
)

In [ ]:
IMG_SIZE = 256
IMG_DIR = "data/"
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() - 2

In [ ]:
class BenGrahamPreprocessing:
    def __init__(self, img_size):
        self.img_size = img_size

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        image_np = np.array(image)

        # Apply OpenCV's gaussian blur
        blurred_image = cv2.GaussianBlur(image_np, (0, 0), self.img_size / 10)

        # Combine the original image with the blurred image
        combined_image = cv2.addWeighted(image_np, 4, blurred_image, -4, 128)

        # Convert the image to a PIL image
        return F.to_pil_image(combined_image)

In [ ]:
class BackgroundRemover:
    def __init__(self, version=1):
        self.version = version
        pass

    def __call__(self, image: pil.Image.Image) -> pil.Image.Image:
        self.width, self.height = image.size
        if self.version == 1:
            return self.crop_black_borders(image)
        else:
            return self.crop_black_borders_v2(image)

    def find_limits(self, img):
        # Find the limits of the eye
        pixels = img.load()

        width, height = self.width, self.height
        half_width = width // 2
        half_height = height // 2
        x_start = 0
        for x in range(width):
            if pixels[x, half_height] != (0, 0, 0):
                x_start = x
                break

        x_end = width - 1
        for x in range(width - 1, -1, -1):
            if pixels[x, half_height] != (0, 0, 0):
                x_end = x
                break

        y_start = 0
        for y in range(height):
            if pixels[half_width, y] != (0, 0, 0):
                y_start = y
                break

        y_end = height - 1
        for y in range(height - 1, -1, -1):
            if pixels[half_width, y] != (0, 0, 0):
                y_end = y
                break

        return x_start, x_end, y_start, y_end

    def get_center_radius(self, x_start, x_end, y_start, y_end):
        # Get the center and radius of the eye
        center_x = (x_start + x_end) // 2
        center_y = (y_start + y_end) // 2
        radius = max((x_end - x_start) // 2, (y_end - y_start) // 2)
        return center_x, center_y, radius

    def crop_eye(self, image, center_x, center_y, radius):
        # Crop the eye from the image using the center and radius
        side = int((2 * radius) / math.sqrt(2))
        half_side = side // 2

        left = max(center_x - half_side, 0)
        up = max(center_y - half_side, 0)
        right = min(center_x + half_side, image.width)
        down = min(center_y + half_side, image.height)

        cropped_image = image.crop((left, up, right, down))
        return cropped_image

    def crop_black_borders(self, image):
        # V1 of eye cropping
        x_start, x_end, y_start, y_end = self.find_limits(image)

        cropped_image = image.crop((x_start, y_start, x_end + 1, y_end + 1))
        return cropped_image

    def crop_black_borders_v2(self, img):
        # V2 of eye cropping
        x_start, x_end, y_start, y_end = self.find_limits(img)

        center_x, center_y, radius = self.get_center_radius(
            x_start, x_end, y_start, y_end
        )

        return self.crop_eye(img, center_x, center_y, radius)

In [ ]:
def define_transform(imgsize, bg_rem_version, custom_prep):
    tranforms = [
        BackgroundRemover(version=bg_rem_version),
        transforms.Resize((imgsize, imgsize)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
        # transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225]), # Normalizing is disabled because it works better without it
    ]
    if custom_prep:
        tranforms.insert(4, BenGrahamPreprocessing(IMG_SIZE))

    return transforms.Compose(tranforms)

In [ ]:
def define_loaders(transform):
    trainset = torchvision.datasets.ImageFolder(root="data/train/", transform=transform)
    trainloader = torch.utils.data.DataLoader(
        trainset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    testset = torchvision.datasets.ImageFolder(root="data/test/", transform=transform)
    testloader = torch.utils.data.DataLoader(
        testset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    return trainloader, testloader

In [ ]:
BATCH_SIZE = 32


def save_preprocessed_images(
    loader, loader_name, img_size, bg_rem_version, custom_prep, num_workers=4
):
    info_str = f"S_{img_size}_R_{bg_rem_version}_C_{custom_prep}"
    # Create the output folder
    output_folder = f"data/prep/{info_str}/{loader_name}"
    os.makedirs(output_folder, exist_ok=True)

    # Create class-specific subfolders
    for class_idx in range(len(loader.dataset.classes)):
        os.makedirs(os.path.join(output_folder, str(class_idx)), exist_ok=True)

    # Define a function to save a single image
    def save_single_image(image, label, index):
        class_folder = str(label.item())
        save_image(
            image,
            os.path.join(output_folder, class_folder, f"image_{index}.png"),
        )

    # Use a ThreadPoolExecutor to save images in parallel
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        image_count = 0
        for i, (images, labels) in tqdm(
            enumerate(loader), desc=f"Processing {loader_name}", total=len(loader)
        ):
            for j in range(images.size(0)):
                # Save the image in a separate thread
                executor.submit(save_single_image, images[j], labels[j], image_count)
                image_count += 1

    print(
        f"Transformed images saved to '{output_folder}' in class-specific subfolders."
    )

In [ ]:
# Define all configurations to process
configs = {"bg_rem_version": [1], "custom_prep": [True], "image_size": [224, 299]}

# Process all combinations
for bg_rem_version, custom_prep, image_size in product(*configs.values()):
    transform = define_transform(image_size, bg_rem_version, custom_prep)
    trainloader, testloader = define_loaders(transform)

    # Process both train and test sets
    for loader, split in [(trainloader, "train"), (testloader, "test")]:
        save_preprocessed_images(
            loader,
            split,
            image_size,
            bg_rem_version,
            custom_prep,
            num_workers=NUM_WORKERS,
        )

In [ ]:
def augment_and_balance_dataset(dataset_path, augmentation_pipeline):
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(dataset_path))  # Sort class names consistently
    biggest_class_size = 0
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_dir):
            for image_name in os.listdir(class_dir):
                image_path = os.path.join(class_dir, image_name)
                image_paths.append(image_path)
                labels.append(class_idx)
        class_indices = [i for i, label in enumerate(labels) if label == class_idx]
        num_samples = len(class_indices)
        if num_samples > biggest_class_size:
            biggest_class_size = num_samples

    target_samples_per_class = int(
        biggest_class_size * 1.25
    )  # Augment to 125% of the biggest class and balance
    for class_idx, class_name in tqdm(enumerate(class_names)):
        class_indices = [i for i, label in enumerate(labels) if label == class_idx]
        num_samples = len(class_indices)

        if num_samples < target_samples_per_class:
            # Augment the class to reach the target number of samples
            images_to_augment = target_samples_per_class - num_samples
            original_image_paths = [image_paths[i] for i in class_indices]

            for i in range(images_to_augment):
                # Randomly select an image from the class to augment
                random_image_path = random.choice(original_image_paths)
                image = cv2.imread(random_image_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert to RGB

                # Apply augmentations
                augmented = augmentation_pipeline(image=image)
                augmented_image = augmented["image"]

                # Save the augmented image
                augmented_image_name = f"{i}_augmented_{len(class_indices)}.png"
                augmented_image_path = os.path.join(
                    dataset_path, class_name, augmented_image_name
                )
                cv2.imwrite(
                    augmented_image_path,
                    cv2.cvtColor(augmented_image, cv2.COLOR_RGB2BGR),
                )  # Save as BGR


augmentation_pipeline = Compose(
    [
        HorizontalFlip(p=0.5),
        VerticalFlip(p=0.5),
        RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        HueSaturationValue(
            hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5
        ),
        GaussianBlur(blur_limit=(3, 7), p=0.3),
    ]
)

base_dir = "data/prep/"

for config in os.listdir(base_dir):
    for split in os.listdir(os.path.join(base_dir, config)):
        split_dir = os.path.join(base_dir, config, split)
        augment_and_balance_dataset(split_dir, augmentation_pipeline)